# 02 — Integrated Business Analytics & Consumer Sentiment

This notebook integrates SHEIN media coverage, public search attention, and consumer review data using sentiment analysis, monthly aggregation, Pearson correlations, one-month lag analysis, period comparisons, and visual reporting.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import CountVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MEDIA_OUTPUT = OUTPUT_DIR / "media_trends"
CONSUMER_OUTPUT = OUTPUT_DIR / "consumer_sentiment"
CORRELATION_OUTPUT = OUTPUT_DIR / "correlation_analysis"
for directory in [MEDIA_OUTPUT, CONSUMER_OUTPUT, CORRELATION_OUTPUT]:
    directory.mkdir(parents=True, exist_ok=True)


## 1. VADER consumer sentiment

Standard thresholds classify reviews as positive (≥ 0.05), neutral, or negative (≤ -0.05). Star ratings remain a separate consumer-evaluation measure.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def sentiment_score(text):
    return analyzer.polarity_scores(str(text))["compound"]

def sentiment_label(score):
    if score >= 0.05: return "positive"
    if score <= -0.05: return "negative"
    return "neutral"

# reviews = pd.read_csv(PROCESSED_DIR / "shein_reviews_clean.csv")
# reviews["sentiment"] = reviews["review_text"].apply(sentiment_score)
# reviews["sentiment_label"] = reviews["sentiment"].apply(sentiment_label)


## 2. Monthly consumer metrics


In [ ]:
def build_monthly_reviews(reviews):
    return (reviews.groupby("month").agg(
        review_count=("review_text", "size"),
        average_rating=("rating", "mean"),
        average_sentiment=("sentiment", "mean")
    ).reset_index())


## 3. Review themes

Positive reviews frequently emphasized quality, price, sizing, ease of use, shipping, delivery, and recommendations. Negative reviews more often emphasized orders, customer service, refunds, returns, money, and delivery problems.


In [ ]:
EXTRA_STOP_WORDS = {"shein", "app", "shopping", "really", "just", "like", "im", "ive", "dont", "didnt", "cant"}

def top_terms(text_series, n=20):
    vectorizer = CountVectorizer(stop_words="english", max_features=5000)
    matrix = vectorizer.fit_transform(text_series.dropna().astype(str))
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    result = pd.DataFrame({"term": terms, "count": counts})
    result = result[~result["term"].isin(EXTRA_STOP_WORDS)]
    return result.sort_values("count", ascending=False).head(n)


## 4. Integrate media, search, and consumer data

The common grain is **month**. Metrics include total headlines, ethics-related volume/share, Google search interest, average star rating, and average VADER sentiment.


In [ ]:
def integrate_sources(monthly_media, trends, monthly_reviews):
    media_trends = monthly_media.merge(trends, on="month", how="left")
    return media_trends.merge(monthly_reviews, on="month", how="left").sort_values("month").reset_index(drop=True)


## 5. Correlation analysis

Pearson correlations evaluate whether media scrutiny, consumer evaluation, and public attention moved together. These are associations, not causal effects.


In [ ]:
def pairwise_correlations(df, relationships):
    results = []
    for x, y in relationships:
        temp = df[[x, y]].dropna()
        r, p = pearsonr(temp[x], temp[y])
        results.append({"x": x, "y": y, "correlation_r": round(r, 3), "p_value": round(p, 4)})
    return pd.DataFrame(results)

consumer_relationships = [
    ("ethics_related_share", "average_rating"),
    ("ethics_related_share", "average_sentiment"),
    ("ethics_related_volume", "average_rating"),
    ("ethics_related_volume", "average_sentiment")
]


## 6. One-month lag analysis

Final project results showed modest negative associations between higher absolute ethics-related media volume and **next-month ratings** (`r = -0.270`, `p = .027`) and **next-month sentiment** (`r = -0.243`, `p = .048`).


In [ ]:
def build_lagged_analysis(integrated_df):
    lagged_df = integrated_df.copy()
    lagged_df["next_month_rating"] = lagged_df["average_rating"].shift(-1)
    lagged_df["next_month_sentiment"] = lagged_df["average_sentiment"].shift(-1)
    relationships = [
        ("ethics_related_share", "next_month_rating"),
        ("ethics_related_share", "next_month_sentiment"),
        ("ethics_related_volume", "next_month_rating"),
        ("ethics_related_volume", "next_month_sentiment")
    ]
    return lagged_df, pairwise_correlations(lagged_df, relationships)


## 7. Media scrutiny vs. search interest

A central final result was a positive association between **ethics-related media share and Google search interest**: **r = 0.450, p < .001**.


In [ ]:
search_relationships = [
    ("total_headlines", "search_interest"),
    ("ethics_related_volume", "search_interest"),
    ("ethics_related_share", "search_interest")
]


## 8. Growth-period comparison

| Period | Avg. Search Interest | Avg. Ethics Share | Avg. Rating | Avg. Sentiment |
|---|---:|---:|---:|---:|
| Pre-Pandemic (2019) | 15.75 | 0.00% | 3.37 | 0.35 |
| Pandemic Growth (2020–2021) | 46.58 | 3.87% | 4.10 | 0.59 |
| Post-Rise Scrutiny (2022–Aug. 2024) | 78.16 | 10.38% | 3.56 | 0.44 |


In [ ]:
def assign_period(month):
    year = int(str(month)[:4])
    if year == 2019: return "Pre-Pandemic (2019)"
    if year in [2020, 2021]: return "Pandemic Growth (2020–2021)"
    return "Post-Rise Scrutiny (2022–Aug 2024)"

def summarize_periods(integrated_df):
    period_df = integrated_df.copy()
    period_df["period"] = period_df["month"].apply(assign_period)
    summary = (period_df[period_df["average_rating"].notna()].groupby("period").agg(
        avg_search_interest=("search_interest", "mean"),
        avg_ethics_share=("ethics_related_share", "mean"),
        avg_rating=("average_rating", "mean"),
        avg_sentiment=("average_sentiment", "mean")
    ).round(2))
    return summary.reindex(["Pre-Pandemic (2019)", "Pandemic Growth (2020–2021)", "Post-Rise Scrutiny (2022–Aug 2024)"])


## 9. Portfolio charts


In [ ]:
def plot_search_vs_ethics(integrated_df):
    df = integrated_df[["ethics_related_share", "search_interest"]].dropna()
    plt.figure(figsize=(8,6))
    plt.scatter(df["ethics_related_share"], df["search_interest"], alpha=0.7)
    m, b = np.polyfit(df["ethics_related_share"], df["search_interest"], 1)
    x_line = np.linspace(df["ethics_related_share"].min(), df["ethics_related_share"].max(), 100)
    plt.plot(x_line, m*x_line+b, linewidth=2)
    plt.xlabel("Ethics-Related Media Share (%)")
    plt.ylabel("Google Search Interest")
    plt.title("Ethics-Related Media Coverage vs. SHEIN Search Interest")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(CORRELATION_OUTPUT / "ethics_media_vs_search_interest.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_period_consumer_response(period_summary):
    plot_df = period_summary.reset_index()
    x = np.arange(len(plot_df)); width = 0.35
    fig, ax1 = plt.subplots(figsize=(9,6))
    ax1.bar(x-width/2, plot_df["avg_rating"], width, label="Average Star Rating")
    ax1.set_ylabel("Average Star Rating"); ax1.set_ylim(0,5)
    ax2 = ax1.twinx()
    ax2.bar(x+width/2, plot_df["avg_sentiment"], width, alpha=0.7, label="Average VADER Sentiment")
    ax2.set_ylabel("Average VADER Sentiment"); ax2.set_ylim(0,0.7)
    ax1.set_xticks(x)
    ax1.set_xticklabels(["Pre-Pandemic\n2019", "Pandemic Growth\n2020–2021", "Post-Rise Scrutiny\n2022–Aug 2024"])
    plt.title("SHEIN Consumer Response Across Growth Periods")
    plt.tight_layout()
    plt.savefig(CONSUMER_OUTPUT / "consumer_response_by_period.png", dpi=200, bbox_inches="tight")
    plt.show()


## Business interpretation

1. **Visibility and favorability are different metrics.** Increased controversy can coincide with higher attention.
2. **Consumer response may lag media scrutiny.** Same-month ratings and sentiment were comparatively resilient, while some weaker next-month associations appeared for absolute ethics-related media volume.
3. **Reputation monitoring should integrate multiple signals.** Media coverage, search behavior, ratings, and review sentiment provide a more complete picture than any one metric alone.

## Limitations

- Google News RSS is a live source rather than a complete historical archive.
- The three sources do not cover identical periods.
- Google Trends reports normalized rather than absolute search volume.
- Ethics classification uses a manually constructed keyword dictionary.
- App reviews are self-selected and do not directly measure purchases or retention.
- VADER may miss sarcasm and context-specific meaning.
- Correlations do not establish causation.
